### Code and data inclusion, and CSV export of DataFrames

This notebook demonstrates how to install local Python packages at build-time, include local Python modules at run-time, and bundle local static data used by the notebook. It also demonstrates generation of a pandas `DataFrame`; like xarray `DataSets`, pandas `DataFrames` are automatically written and catalogued by xcengine for EOAP stage-out when the generated Docker image is run in EOAP mode.

#### Define parameters and configure xcengine

The first cell is the parameters cell, which defines the since input parameter of this notebook (`factor`) and configures xcengine with an `xcengine_config` dictionary.

In [1]:
factor = 2

xcengine_config = dict(
    workflow_id="inclusions",
    environment_file="../environment.yml",
    container_image_tag="inclusions:1",
    include_directory=True,  # include the whole parent directory of the notebook
    build_includes=["../mylocalpackage"]  # install these local packages in the build
)

#### Import published packages

These are the published packages used by the notebook, which will be
installed from conda-forge as specified in `environment.yml`.

In [2]:
import csv
import functools
import pathlib
import pandas as pd

#### Import a run-time inclusion

Import a function from `mymodule.py`, which is bundled as a run-time inclusion with the notebook because we configured `include_directory=True` above.

In [3]:
from mymodule import addone

#### Import a build-time inclusion

Import a function from `mylocalpackage`, which is configured using `build_includes`. This means that `mylocalpackage` is not bundled with the notebook, but instead is pre-installed in the Python environment when `xcetool` builds the Docker image.

In [4]:
from mylocalpackage import multiply

#### Read an included data file

Reading local data works slightly differently in the notebook and the generated EOAP script. In the notebook, we can usually assume that the current directory (CWD) is the notebook's directory and look for data there. In an EOAP, the current directory is usually not the same as the script's directory, so we have to explicitly define the data directory relative to the script's own path.

Like `mymodule.py`, the data file `input-data.csv` is bundled with the notebook because we configured `include_directory=True` in the parameters cell.

In [5]:
try:
    # Find the script's parent directory, if we're running as a script.
    datadir = pathlib.Path(__file__).parent
except NameError:
    # If __file__ is not defined, assume we're running in a notebook
    # and look for the data file in the current working directory.
    datadir = pathlib.Path.cwd()

Now we read the data file into a pandas `DataFrame` using the `datadir` variable that we set above.

In [6]:
df = pd.read_csv(datadir / "input-data.csv")

df

,1
0,3
1,7
2,13
3,23
4,42


Process the data using the functions we imported from our local code and the parameter `factor`, writing the output into a new data frame called `result`. We use both the imported functions and the parameter `factor` in the calculation.

In [7]:
multiply_by_factor = functools.partial(multiply, factor)
result = df.apply(multiply_by_factor).apply(addone)

result

,1
0,7
1,15
2,27
3,47
4,85


The new data frame `result` will be automatically found and written as CSV by xcengine. In EOAP mode, it will also be included in a stage-out catalogue. So we don't need to do anything special to export our result here.